In [1]:
import numpy as np
import pandas as pd
import h5py

import pycbc.conversions, pycbc.distributions, pycbc.waveform, pycbc.filter, pycbc.types, pycbc.psd, pycbc.fft

from tqdm import tqdm
import datetime
import multiprocessing
import uuid
from argparse import ArgumentParser
import logging

/work/yifanwang/ecc/env913ecc/lib/python3.11/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal


In [40]:
class GenWaveform(object):
    '''Waveform Generator
    '''
    def __init__(self, buffer_length, sample_rate, f_lower):
        self.f_lower = f_lower
        self.delta_f = 1.0 / buffer_length
        tlen = int(buffer_length * sample_rate) # buffer length x sample_rate
        self.flen = tlen // 2 + 1

        #psd is hard coded to O3 psd
        psd = pycbc.psd.read.from_txt('/work/yifanwang/ecc/templatebank/o3psd.txt', 
            self.flen, self.delta_f, self.f_lower, is_asd_file = False)
        
        self.kmin = int(f_lower * buffer_length)
        self.w = ((1.0 / psd[self.kmin:-1]) ** 0.5).astype(np.float32)
        
        qtilde = pycbc.types.zeros(tlen, np.complex64) # correlation in Fourier domain
        q = pycbc.types.zeros(tlen, np.complex64) # correlation
        self.qtilde_view = qtilde[self.kmin:self.flen - 1]
        self.ifft = pycbc.fft.IFFT(qtilde, q)
        
        # the maximum is around 0
        self.md = q._data[-100:]
        self.md2 = q._data[0:100] 

    def generate(self, **kwds):
        '''Return normalized hp
        '''
        if kwds['approximant'] in pycbc.waveform.fd_approximants():  
            hp, _ = pycbc.waveform.get_fd_waveform(delta_f = self.delta_f, **kwds)
        else:
            dt = 1.0 / self.sample_rate
            hp = pycbc.waveform.get_waveform_filter(
                        pycbc.types.zeros(self.flen, dtype=np.complex64),
                        delta_f=self.delta_f,
                        delta_t=dt,
                        f_lower=self.f_lower,
                        **kwds)
        
        hp.resize(self.flen)
        hp = hp.astype(np.complex64)
        
        hp[self.kmin:-1] *= self.w
        s = pycbc.filter.sigmasq(hp, low_frequency_cutoff=self.f_lower)
        hp /= s**0.5 
        
        hp.params = kwds
        hp.s = s

        return hp

    def match(self, hp, hc):
        hp.view = hp[self.kmin:-1]
        hc.view = hc[self.kmin:-1]
        pycbc.filter.correlate(hp.view, hc.view, self.qtilde_view)
        self.ifft.execute()
        m = max(abs(self.md).max(), abs(self.md2).max())
        return m * 4.0 * self.delta_f

    def overlap(self, hp, hc):
        o = hp.inner(hc)
        return o * 4.0 * self.delta_f

def wf_wrapper(p):
    index = p['index']
    try:
        hp = gen.generate(**p)
        return index, hp
    except Exception:
        return index, None
    
def failed_wf(p):
    index = p['index']
    try:
        hp = gen.generate(**p)
        return None
    except Exception:
        return index

In [68]:
with h5py.File('/work/yifanwang/ecc/templatebank/newbank/bank1000/bank1000duration.hdf') as f:
    df_bank = pd.DataFrame(
       {'mass1': f['mass1'][:],
        'mass2': f['mass2'][:],
        'eccentricity': f['eccentricity'][:],
        'rel_anomaly': f['rel_anomaly'][:],
        'spin1z': f['spin1z'][:],
        'spin2z': f['spin2z'][:],
        'approximant': f['approximant'][:].astype('str'),
        'f_lower': f['f_lower'][:],
        'template_duration': f['template_duration'][:]}
    )

In [69]:
gen = GenWaveform(buffer_length = 32, sample_rate = 2048, f_lower = 20)

In [70]:
df_bank['index'] = df_bank.index

In [71]:
df_bank

,mass1,mass2,eccentricity,rel_anomaly,spin1z,spin2z,approximant,f_lower,template_duration,index
0,18.226825,66.423883,0.290814,3.753081,0.428049,0.123707,SEOBNRv5E,20.0,0.482910,0
1,90.081933,52.653331,0.280265,3.316337,0.260539,0.210497,SEOBNRv5E,20.0,0.158203,1
2,38.133665,30.978496,0.212317,4.666902,-0.060656,-0.090567,SEOBNRv5E,20.0,0.544434,2
3,61.233045,93.431521,0.211070,4.531660,0.346991,0.181205,SEOBNRv5E,20.0,0.145508,3
4,81.009968,31.343283,0.146425,2.901016,0.493283,0.036169,SEOBNRv5E,20.0,0.352051,4
...,...,...,...,...,...,...,...,...,...,...
995,5.024807,5.084112,0.252812,4.903667,-0.399093,-0.247737,SEOBNRv5E,20.0,14.809570,995
996,5.038961,5.033609,0.290182,1.950168,0.479835,0.438965,SEOBNRv5E,20.0,14.333008,996
997,5.011231,5.005721,0.235693,2.690773,0.349650,0.362697,SEOBNRv5E,20.0,16.009766,997
998,5.018156,5.016282,0.244095,4.290929,-0.032637,-0.117191,SEOBNRv5E,20.0,15.423828,998


In [72]:
failed_index = []
parlist = ['index', 'approximant', 'f_lower', 'mass1', 'mass2', 'spin1z', 'spin2z', 'eccentricity', 'rel_anomaly']
with multiprocessing.Pool(64) as pool:
    for return_i in pool.imap_unordered(
        failed_wf,
        ({k: df_bank.loc[idx, k] for k in parlist} for idx in tqdm(df_bank.index))
    ):
        failed_index += [return_i]

  0%|          | 0/1000 [00:00<?, ?it/s]WARNING:pyseobnr.models.SEOBNRv5EHM:The predicted starting separation of the system (r_start = 9.916332016502114) is below the minimum starting separation allowed by the model (r_start_min = 10.0). Integrating backwards in time a set of secular evolution equations to obtain a prediction for the starting separation larger than this minimum. This will change the starting input values of the system.
ERROR:pyseobnr.eob.dynamics.integrate_ode_ecc:Internal function call failed: Input domain error. The time length of the dynamics is below 200 M. Aborting waveform generation since this could cause non-physical waveforms or problems in subsequent steps of the model. Please, review the physical sense of the input parameters.
ERROR:pyseobnr.models.SEOBNRv5EHM:Waveform generation failed for q = 1.0153946499549098, chi_1 = 0.02416545098403336, chi_2 = 0.4209184089416159, omega_avg = 0.04731693113492515, omega_inst = 0.02952502262615315, eccentricity = 0.28746

In [73]:
failed_index = [i for i in failed_index if i != None]

In [74]:
failed_index

[14]

In [75]:
update_df = df_bank.drop(failed_index)

In [76]:
with h5py.File('removefailbank1000.hdf','w') as f_write:
    with h5py.File('/work/yifanwang/ecc/templatebank/newbank/bank1000/bank1000duration.hdf','r') as f_bank:
        for k in f_bank.keys():
            if k=='approximant':
                print('convert to bytes')
                f_write[k] = update_df[k].values.astype('bytes')
            else:
                f_write[k] = update_df[k].values

convert to bytes
